The notebook is a PoC to retrieve information from a corpus to create a knowledge graph.

In [1]:
import json
import ollama
from sentence_transformers import SentenceTransformer
import os
import hashlib
import sys
sys.path.insert(1, "src/utils/")
from prepare_graph import get_node_id

In [2]:
text = """
Le président Emmanuel Macron a rencontré Olaf Scholz à Berlin
afin de renforcer la coopération européenne sur les questions énergétiques.
"""

# Prompt pour Ollama

In [3]:
with open(os.path.join("src", "prepare_llm_output_graph.json"), encoding="utf-8") as json_file:
    example_json = json.load(json_file)

In [4]:
prompt_template = ""
with open(os.path.join("src", "utils", "prompt_RAG.txt")) as filin:
    for line in filin:
        prompt_template += line

In [5]:
prompt = prompt_template.format(
    example_json=example_json,
    text=text
)

# Appel au LLM

In [6]:
response = ollama.chat(
    model="qwen2.5:7b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    options={
        "temperature": 0
    }
)

In [7]:
# convert llm's output as JSON
data = json.loads(response["message"]["content"])

The LLM extracts the information. Now we need to add the ids

In [9]:
# 1. create IDs
for entity in data["entities"]:
    entity["id"] = get_node_id(entity)

# 2. Build index
entity_index = {
    entity["name"]: entity
    for entity in data["entities"]
}

# 3. Relationship
for relation in data["relations"]:
    relation["source_id"] = get_node_id(entity_index[relation["source"]])
    relation["target_id"] = get_node_id(entity_index[relation["target"]])

In [11]:
graph = {
    "nodes": [],
    "relationships": []
}

for entity in data["entities"]:
    graph["nodes"].append({
        "id": entity["id"],
        "label": entity["type"],      # Label Neo4j
        "properties": {
            "name": entity["name"]
        }
    })

for relation in data["relations"]:
    graph["relationships"].append({
        "type": relation["relation"],
        "source": relation["source_id"],
        "target": relation["target_id"],
        "properties": {
            "evidence": relation["evidence"],
            "confidence": relation["confidence"]
        }
    })

In [12]:
graph

{'nodes': [{'id': 'Person_795135bdbfd5fdeb8ea31b549c4cd04d4f9c17899e96ffd81a4d5c74d796932b',
   'label': 'Person',
   'properties': {'name': 'Emmanuel Macron'}},
  {'id': 'Person_7cf2dcc30d3aa074ce7408ccfda92d570e9ebc66f3f3a2c14ad98461cf11902b',
   'label': 'Person',
   'properties': {'name': 'Olaf Scholz'}},
  {'id': 'Event_96d141a9b5904ec92912212ae19a35e04a7db3b0cd415b5b2883ca5f2c0be350',
   'label': 'Event',
   'properties': {'name': 'Rencontre diplomatique'}},
  {'id': 'City_b99770cd89b8870c4e168b4e52a8f787b64a4a2fc9ef992bf093af11c7e6a055',
   'label': 'City',
   'properties': {'name': 'Berlin'}},
  {'id': 'Theme_7953a413bb3ca5489902d5ba93d71753e03f652380ab1937088b85570e55293e',
   'label': 'Theme',
   'properties': {'name': 'Énergie'}}],
 'relationships': [{'type': 'MET_WITH',
   'source': 'Person_795135bdbfd5fdeb8ea31b549c4cd04d4f9c17899e96ffd81a4d5c74d796932b',
   'target': 'Person_7cf2dcc30d3aa074ce7408ccfda92d570e9ebc66f3f3a2c14ad98461cf11902b',
   'properties': {'evidence': '

# Load embedding model

Next step is to add embedding to the node. By this way we can make text similarity querry 

In [13]:
embedding_model = SentenceTransformer("BAAI/bge-m3")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [14]:
EMBEDDING_LABELS = {"Theme", "Event", "Speech", "Chunk"}

for node in graph["nodes"]:
    if node["label"] in EMBEDDING_LABELS:
        embedding = embedding_model.encode(node["properties"]["name"])
        node["properties"]["embedding"] = embedding.tolist()

In [16]:
graph

{'nodes': [{'id': 'Person_795135bdbfd5fdeb8ea31b549c4cd04d4f9c17899e96ffd81a4d5c74d796932b',
   'label': 'Person',
   'properties': {'name': 'Emmanuel Macron'}},
  {'id': 'Person_7cf2dcc30d3aa074ce7408ccfda92d570e9ebc66f3f3a2c14ad98461cf11902b',
   'label': 'Person',
   'properties': {'name': 'Olaf Scholz'}},
  {'id': 'Event_96d141a9b5904ec92912212ae19a35e04a7db3b0cd415b5b2883ca5f2c0be350',
   'label': 'Event',
   'properties': {'name': 'Rencontre diplomatique',
    'embedding': [0.01992581970989704,
     -0.014285228215157986,
     -0.024599555879831314,
     -0.006653469987213612,
     -0.026447521522641182,
     0.01005741860717535,
     0.04769282788038254,
     -0.03637024015188217,
     0.050031621009111404,
     0.009416828863322735,
     0.05916305258870125,
     0.01343879196792841,
     0.005379930138587952,
     0.011026491411030293,
     -0.019420843571424484,
     0.009112360887229443,
     0.01480130571871996,
     0.00020829829736612737,
     0.03434721380472183,
     0.

In [ ]:
embedding_model = SentenceTransformer("BAAI/bge-m3")
text = """
Le président Emmanuel Macron a rencontré Olaf Scholz à Berlin
afin de renforcer la coopération européenne sur les questions énergétiques.
"""

# Generate embedding
embedding = embedding_model.encode(text).tolist()
print(f"Dimension de l'embedding : {len(embedding)}")